# P02｜像素腦讀論文：YOLO

論文：Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi, "You Only Look Once: Unified, Real-Time Object Detection", CVPR 2016。https://arxiv.org/abs/1506.02640

這本筆記本有兩部分：
1. 用 Ultralytics 的開源套件跑 2026 年的 YOLO26，看看現在的 YOLO 長什麼樣子（不是論文的原始程式）。
2. **像素腦的示意版**：照論文的想法（7×7 格、每格 2 個框、λcoord = 5、λnoobj = 0.5、寬高取平方根、信心等於 IOU）自己寫一個很小的 YOLO，在像素數字上從零訓練。最後的輸出層改成卷積（論文是全連接層）。這是示意，不是論文的程式；論文的原始程式是 C 寫的 Darknet。

全部用 CPU 就能跑，訓練大約幾分鐘。

## 0. 準備：安裝套件、下載兩張貓咪照片（都是 CC0：一隻貓 AdinaVoicu、兩隻小貓 Irshath 25，Wikimedia Commons）

In [ ]:
!pip -q install ultralytics

In [ ]:
import os, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch, torch.nn as nn
from torchvision.ops import box_iou, nms
from sklearn.datasets import load_digits
from PIL import Image

if not os.path.exists("cat.png"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep02-edges/cat.png", "cat.png")
if not os.path.exists("cats.jpg"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/p02-yolo/cats.jpg", "cats.jpg")

## 1. 先看成果：現在的 YOLO 看貓咪照片

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")                 # Ultralytics 2026 年的 YOLO26，不是論文的原始程式
for f in ["cat.png", "cats.jpg"]:
    r = model(f, verbose=False)[0]
    print(f, f"{r.speed['inference']:.0f} 毫秒")
    for b in r.boxes:
        print(" ", r.names[int(b.cls)], f"{float(b.conf):.0%}", [round(v) for v in b.xyxy[0].tolist()])
    plt.imshow(r.plot()[..., ::-1]); plt.axis("off"); plt.show()

## 2. 示意版：做一堆有像素數字的圖

每張圖 112×112，切成 7×7 格，每格 16 像素。訓練資料裡，一格最多只有一個數字的中心。

In [ ]:
S, B, C, SIZE = 7, 2, 10, 112             # 7×7 格、每格 2 個框、10 類（數字 0 到 9）
CELL = SIZE // S                           # 每格 16 像素
digits = load_digits()                     # 1,797 張 8×8 的像素數字
rng = np.random.default_rng(0)

def make_scene(n_max=4, centers=None):
    img = np.zeros((SIZE, SIZE), np.float32)
    boxes, used = [], set()
    n = len(centers) if centers else rng.integers(1, n_max + 1)
    while len(boxes) < n:
        k = rng.integers(len(digits.images))
        if centers:                        # 指定中心位置（第 8 步的實驗用），大小固定 24 像素
            s = 24; cx, cy = centers[len(boxes)]
        else:
            s = int(rng.integers(3, 6)) * 8                    # 24、32、40 像素
            cx, cy = rng.integers(s // 2, SIZE - s // 2, size=2)
            if (cy // CELL, cx // CELL) in used:
                continue                   # 訓練資料：一格最多一個物體的中心
        x0, y0 = int(cx - s // 2), int(cy - s // 2)
        patch = np.kron(digits.images[k] / 16, np.ones((s // 8, s // 8)))
        img[y0:y0 + s, x0:x0 + s] = np.maximum(img[y0:y0 + s, x0:x0 + s], patch)
        boxes.append([digits.target[k], cx / SIZE, cy / SIZE, s / SIZE, s / SIZE])
        used.add((cy // CELL, cx // CELL))
    return img, np.array(boxes, np.float32)

train = [make_scene() for _ in range(12000)]
test = [make_scene() for _ in range(500)]
print(len(train), len(test), "張；第一張有", len(test[0][1]), "個數字")

## 3. 網路：卷積抽特徵，最後輸出 7×7×20

每格 2 個框 × 5 個數（x、y、√w、√h、信心）＋ 10 類的機率 = 20。論文是 20 類，所以是 7×7×30。

論文最後兩層是全連接層。像素腦一開始也照做，結果練過的題目答得很好、沒看過的就差很多（資料太少，全連接層每一格都要各學一次）；所以示意版改成每一格共用同一組權重的卷積（後來 YOLO 第二版也拿掉了全連接層）。另外加了 BatchNorm、用 Adam，讓 CPU 幾分鐘內練得完。

In [ ]:
class TinyYOLO(nn.Module):                # 卷積抽特徵 → 每一格輸出 B×5+C 個數 → 7×7×20
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.1), nn.MaxPool2d(2))
        self.conv = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128))   # 112 → 7
        self.head = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.LeakyReLU(0.1),
                                  nn.Conv2d(128, B * 5 + C, 1))   # 論文這裡是全連接層；示意版改成每格共用的卷積
    def forward(self, x):                  # 最後一層是線性的，和論文一樣
        return self.head(self.conv(x)).permute(0, 2, 3, 1)

net = TinyYOLO()
print(net(torch.zeros(1, 1, SIZE, SIZE)).shape, sum(p.numel() for p in net.parameters()), "個參數")

## 4. 論文的損失函數

中心落在哪一格，那一格就負責；兩個框裡跟答案 IOU 比較高的那個負責。

In [ ]:
def to_xyxy(p, i, j):                       # 格子裡的 (x, y, √w, √h) → 整張圖的框
    cx, cy = (j + p[..., 0]) / S, (i + p[..., 1]) / S
    w, h = p[..., 2].clamp(0) ** 2, p[..., 3].clamp(0) ** 2
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)

def make_target(boxes):                    # 中心落在哪一格，就由那一格負責
    t = torch.zeros(S, S, 5 + C)           # 有沒有物體、x、y、√w、√h、10 類
    for c, x, y, w, h in boxes.tolist():
        i, j = min(int(y * S), S - 1), min(int(x * S), S - 1)
        t[i, j, :5] = torch.tensor([1, x * S - j, y * S - i, w ** 0.5, h ** 0.5])
        t[i, j, 5 + int(c)] = 1
    return t

I, J = torch.meshgrid(torch.arange(S), torch.arange(S), indexing="ij")

def yolo_loss(out, t, l_coord=5.0, l_noobj=0.5):     # 論文的 λcoord = 5、λnoobj = 0.5
    obj = t[..., 0] == 1
    truth = to_xyxy(t[..., 1:5], I, J)
    pred = torch.stack([out[..., b * 5: b * 5 + 5] for b in range(B)], 3)    # N×7×7×2×5
    ious = torch.stack([box_iou_pair(to_xyxy(pred[..., b, :4], I, J), truth) for b in range(B)], -1).detach()
    resp = nn.functional.one_hot(ious.argmax(-1), B).bool() & obj[..., None]  # IOU 比較高的那個框負責
    p, tt = pred[resp], t[resp.any(-1)]
    loss = l_coord * ((p[:, :4] - tt[:, 1:5]) ** 2).sum()            # 位置與寬高（寬高取平方根）
    loss = loss + ((p[:, 4] - ious[resp]) ** 2).sum()                 # 信心要等於 IOU
    loss = loss + l_noobj * (pred[..., 4][~resp] ** 2).sum()          # 不負責的框：信心要是 0
    loss = loss + ((out[..., B * 5:][obj] - t[..., 5:][obj]) ** 2).sum()   # 類別機率
    return loss / len(out)

def box_iou_pair(a, b):                    # 兩組框一對一算 IOU
    lt, rb = torch.max(a[..., :2], b[..., :2]), torch.min(a[..., 2:], b[..., 2:])
    inter = (rb - lt).clamp(0).prod(-1)
    area = lambda x: (x[..., 2:] - x[..., :2]).clamp(0).prod(-1)
    return inter / (area(a) + area(b) - inter + 1e-9)

## 5. 訓練

In [ ]:
torch.manual_seed(0)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, [8, 11], 0.1)    # 後面降低學習率，和論文一樣分段
X = torch.tensor(np.stack([im for im, _ in train]))[:, None]
T = torch.stack([make_target(bx) for _, bx in train])
t0 = time.time()
for epoch in range(1, 13):
    net.train()
    for k in torch.randperm(len(train)).split(64):
        loss = yolo_loss(net(X[k]), T[k])
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    if epoch % 3 == 0:
        print(f"第 {epoch} 輪：loss {loss.item():.2f}，已花 {time.time() - t0:.0f} 秒")

## 6. 一次看完整張圖：98 個框 → 分數門檻 → NMS

In [ ]:
def detect(img, thresh=0.2, use_nms=True):
    net.eval()
    with torch.no_grad():
        out = net(torch.tensor(img)[None, None])[0]
    i, j = torch.meshgrid(torch.arange(S), torch.arange(S), indexing="ij")
    boxes = torch.cat([to_xyxy(out[..., b * 5: b * 5 + 4], i, j).view(-1, 4) for b in range(B)])
    conf = torch.cat([out[..., b * 5 + 4].reshape(-1) for b in range(B)])
    cls_p = out[..., B * 5:].reshape(-1, C).repeat(B, 1)
    score, cls = (conf[:, None] * cls_p).max(1)            # 每個框的分數 = 信心 × 類別機率
    keep = score > thresh
    boxes, score, cls = boxes[keep], score[keep], cls[keep]
    if use_nms:                                            # 同一類重疊太多的框，只留分數最高的
        k = nms(boxes, score, 0.5)
        boxes, score, cls = boxes[k], score[k], cls[k]
    return boxes * SIZE, score, cls

def show(img, dets, ax):
    ax.imshow(img, cmap="gray"); ax.axis("off")
    for (x0, y0, x1, y1), s, c in zip(*dets):
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow"))
        ax.text(x0, y0 - 1, f"{int(c)} {float(s):.2f}", color="yellow", fontsize=8)

img = test[1][0]
print("98 個框裡分數超過 0.2 的，NMS 之前：", len(detect(img, use_nms=False)[0]), "個，之後：", len(detect(img)[0]), "個")
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (im, _) in zip(axes, test[1:5]):
    show(im, detect(im), ax)
plt.show()

## 7. 考試：500 張沒看過的圖

框和答案的 IOU 至少 0.5、而且類別對，才算找到。

In [ ]:
def check(im, truth):                       # 框和答案的 IOU 至少 0.5、類別也對，才算找到
    boxes, score, cls = detect(im)
    tb = torch.tensor(np.c_[truth[:, 1:3] - truth[:, 3:] / 2, truth[:, 1:3] + truth[:, 3:] / 2]) * SIZE
    used, wrong = set(), 0
    for b, c in zip(boxes, cls):
        iou = box_iou(b[None], tb)[0]
        k = int(iou.argmax())
        if iou[k] >= 0.5 and int(truth[k, 0]) == int(c) and k not in used:
            used.add(k)
        else:
            wrong += 1
    return len(used), wrong

t0 = time.time()
res = np.array([check(im, truth) for im, truth in test])
ms = (time.time() - t0) / len(test) * 1000
total = sum(len(truth) for _, truth in test)
print(f"{total} 個數字：找到 {res[:, 0].sum()} 個，漏掉 {total - res[:, 0].sum()} 個，多框或認錯 {res[:, 1].sum()} 個")
print(f"每張圖約 {ms:.1f} 毫秒")

## 8. 咦？故意把 2 個數字擠在同一格

論文自己承認的限制：每一格只有一組類別、兩個框，擠在一起的小東西（像一群鳥）會漏掉。同樣兩個數字，中心放在不同格、和擠在同一格，比比看。

In [ ]:
apart, crowd = [], []
for _ in range(100):
    apart.append(check(*make_scene(centers=[(40, 40), (72, 72)]))[0])   # 中心在不同格
    crowd.append(check(*make_scene(centers=[(49, 49), (62, 62)]))[0])   # 中心擠在同一格
print("兩個數字在不同格：平均找到", np.mean(apart), "個")
print("兩個數字擠在同一格：平均找到", np.mean(crowd), "個；兩個都找到的有", sum(c == 2 for c in crowd), "張")
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, c in zip(axes, [[(40, 40), (72, 72)], [(49, 49), (62, 62)]]):
    im, truth = make_scene(centers=c); show(im, detect(im), ax)
plt.show()

## 🧪 自己玩玩看
- 把 `yolo_loss` 的 `l_noobj` 改成 1，重新訓練，多框的數量會怎樣變？
- 把 `detect` 的 `use_nms` 改成 `False`，畫出來會怎樣？
- 把 `S` 改成 14（每格 8 像素），擠在一起的數字找得比較到嗎？
- 第 1 步換成你自己的照片試試看。

像素腦 @pixelbraintw